In [10]:

from pathlib import Path

from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from langchain.tools import tool
from langgraph.checkpoint.memory import MemorySaver

from utils.std_model import base_model

llm = base_model()


@tool
def remove_file(path: str) -> str:
    """Delete a file from the filesystem."""
    return f"Deleted {path}"


@tool
def fetch_file(path: str) -> str:
    """Read a file from the filesystem."""
    return f"Contents of {path}"


@tool
def notify_email(to: str, subject: str, body: str) -> str:
    """Send an email."""
    return f"Sent email to {to}"


# Checkpointer is REQUIRED for human-in-the-loop
checkpointer = MemorySaver()
backend = FilesystemBackend(root_dir=Path(".").resolve(), virtual_mode=True)

agent = create_deep_agent(
    model=llm,
    tools=[remove_file, fetch_file, notify_email],
    backend=backend,
    interrupt_on={
        "remove_file": True,  # Default: approve, edit, reject, respond
        "fetch_file": False,  # No interrupts needed
        "notify_email": {"allowed_decisions": ["approve", "reject"]},  # No editing
    },
    checkpointer=checkpointer,  # Required!
)

from langchain_core.utils.uuid import uuid7
from langgraph.types import Command

# Create config with thread_id for state persistence
config = {"configurable": {"thread_id": str(uuid7())}}

# Invoke the agent
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Delete the file temp.txt"}]},
    config=config,
    version="v2",
)

# Check if execution was interrupted
if result.interrupts:
    # Extract interrupt information
    interrupt_value = result.interrupts[0].value
    action_requests = interrupt_value["action_requests"]
    review_configs = interrupt_value["review_configs"]

    # Create a lookup map from tool name to review config
    config_map = {cfg["action_name"]: cfg for cfg in review_configs}

    # Display the pending actions to the user
    for action in action_requests:
        review_config = config_map[action["name"]]
        print(f"Tool: {action['name']}")
        print(f"Arguments: {action['args']}")
        print(f"Allowed decisions: {review_config['allowed_decisions']}")

    # Get user decisions (one per action_request, in order)
    decisions = [
        {"type": "approve"}  # User approved the deletion
    ]

    # Resume execution with decisions
    result = agent.invoke(
        Command(resume={"decisions": decisions}),
        config=config,  # Must use the same config!
        version="v2",
    )

# Process final result
print(result.value["messages"][-1].content)

Tool: remove_file
Arguments: {'path': '/temp.txt'}
Allowed decisions: ['approve', 'edit', 'reject', 'respond']
Deleted `/temp.txt`.


In [5]:

import os
from pathlib import Path

from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from langchain.tools import tool
from langgraph.checkpoint.memory import MemorySaver

from utils.std_model import base_model

llm = base_model()

# Checkpointer is REQUIRED for human-in-the-loop
checkpointer = MemorySaver()
backend = FilesystemBackend(root_dir=Path(".").resolve(), virtual_mode=True)

@tool
def file_delete(file_path: str) -> str:
  """Delete a file from the filesystem."""
  try:
      resolved = backend._resolve_path(file_path)
  except ValueError as e:
      return f"Error: {e}"
  if not resolved.exists():
      return f"Error: no such file: {file_path}"
  try:
      os.remove(str(resolved))
      return f"Deleted: {file_path}"
  except Exception as e:
      return f"Error: {e}"

agent = create_deep_agent(
    model=llm,
    backend=backend,
    interrupt_on={
        "write_file": True,  # Default: approve, edit, reject, respond
        "read_file": False,  # No interrupts needed
        "file_delete": {"allowed_decisions": ["approve", "reject"]},  # No editing
    },
    tools=[file_delete],
    checkpointer=checkpointer,  # Required!
)

from langchain_core.utils.uuid import uuid7
from langgraph.types import Command

# Create config with thread_id for state persistence
config = {"configurable": {"thread_id": str(uuid7())}}

# Invoke the agent
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Delete the file temp.txt"}]},
    # {"messages": [{"role": "user", "content": "create the empty file temp.txt in current directory and it's content is nihao kemengjian"}]},
    config=config,
    version="v2",
)

# Check if execution was interrupted
if result.interrupts:
    # Get user decisions (one per action_request, in order)
    decisions = [
        {"type": "approve"}  # User approved the deletion
    ]

    # Resume execution with decisions
    result = agent.invoke(
        Command(resume={"decisions": decisions}),
        config=config,  # Must use the same config!
        version="v2",
    )

    print(result)

GraphOutput(value={'messages': [HumanMessage(content='Delete the file temp.txt', additional_kwargs={}, response_metadata={}, id='86ec6ce3-4f6f-4ee4-b48e-882630cd400e'), AIMessage(content='', additional_kwargs={'refusal': None, 'reasoning_content': 'The user wants to delete a file called temp.txt. I need to find its absolute path first since `file_delete` requires an absolute path.'}, response_metadata={'token_usage': {'completion_tokens': 75, 'prompt_tokens': 6016, 'total_tokens': 6091, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 29, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 5888}, 'prompt_cache_hit_tokens': 5888, 'prompt_cache_miss_tokens': 128}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': 'bd0a52da-6a58-41df-94be-3af1b342e316', 'finish_reason': 'tool_calls', 'logprobs